# Week 8 — nanoGPT 风格 Refactor + Kaggle 欺诈 (MVP v0.3 v2)

**目标**：
1. 把 W7 的 Encoder 重写成 nanoGPT 风格（CausalSelfAttention / Block / Transformer，< 150 行）
2. 加 `causal: bool` flag——本 notebook 默认 `False`（异常检测需双向上下文）
3. 在 Kaggle 信用卡欺诈真实数据上打赢 MLP baseline
4. 用 `causal=True` 对照实验，体会 GPT vs BERT 的差异

**产出**：
- nanoGPT-compact 模型（`CausalSelfAttention + Block + Transformer`）
- Kaggle 序列构造（按时间切分、无泄露）
- 与同输入 MLP baseline 对比
- GPT(causal) vs BERT(bidirectional) 对照实验
- checkpoint `PROJECT_ROOT/models/transformer_v2.pt`

In [ ]:
# ── Bootstrap (Colab + local) ──────────────────────────────────────
# Env detect → Drive mount → Kaggle creds → reproducibility
import os, sys, pathlib, random

IN_COLAB = 'google.colab' in sys.modules
DRIVE_FOLDER = 'LLM/transformer-anomaly'  # edit if your Drive subfolder differs

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    PROJECT_ROOT = pathlib.Path(f'/content/drive/MyDrive/{DRIVE_FOLDER}')
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    for sub in ('data', 'models', 'checkpoints'):
        (PROJECT_ROOT / sub).mkdir(exist_ok=True)
    os.chdir(PROJECT_ROOT)

    # Kaggle credentials via Colab Secrets (left sidebar 🔑 icon).
    # Add KAGGLE_USERNAME and KAGGLE_KEY there; never upload kaggle.json.
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    except Exception as e:
        print(f'[bootstrap] Kaggle secrets not configured: {e}')
else:
    PROJECT_ROOT = pathlib.Path.cwd()

# Reproducibility
import numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[bootstrap] env={"colab" if IN_COLAB else "local"}  device={device}')
print(f'[bootstrap] project_root={PROJECT_ROOT}')

In [ ]:
import math, json, time
import numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (average_precision_score, roc_auc_score, roc_curve,
                             precision_recall_curve, confusion_matrix)
from sklearn.preprocessing import StandardScaler
sns.set_theme(style='whitegrid', palette='Set2')
print('torch', torch.__version__, '| device', device)

## 1. 下载 Kaggle `creditcardfraud`

沿用 W1 的套路：bootstrap 已经把 KAGGLE 凭证放到 env 里，直接 `kaggle datasets download`。

In [ ]:
DATA_DIR = PROJECT_ROOT / 'data'
CSV = DATA_DIR / 'creditcard.csv'

if not CSV.exists():
    try:
        import kaggle  # noqa: F401
    except ImportError:
        os.system('pip install -q kaggle')
    DATA_DIR.mkdir(exist_ok=True, parents=True)
    ret = os.system(f'kaggle datasets download -d mlg-ulb/creditcardfraud -p {DATA_DIR} --unzip -q')
    if ret != 0:
        raise RuntimeError('kaggle download failed — check KAGGLE_USERNAME / KAGGLE_KEY')

df = pd.read_csv(CSV)
print('shape', df.shape, '| fraud rate', df['Class'].mean())
df.head()

In [ ]:
# quick sanity: Time is seconds from first tx
print('time span (days):', df['Time'].max() / 86400)
print('fraud rows:', df['Class'].sum(), '| non-fraud:', (df["Class"] == 0).sum())

## 2. nanoGPT-compact 模型（< 150 行）

结构照 Karpathy `nanoGPT/model.py`：
- `CausalSelfAttention`：一次性 `qkv` linear + 头拆分 + 可选 `tril` mask
- `Block`：`x = x + attn(ln1(x)); x = x + mlp(ln2(x))`（Pre-LN）
- `Transformer`：token proj + PE + N 个 Block + 分类头

关键 flag：`causal: bool`。`True` 是 GPT，`False` 是 BERT 风格。

In [ ]:
# =============== nanoGPT-style compact Transformer (<150 lines) ===============
class GPTConfig:
    def __init__(self, n_feat, d_model=96, n_head=4, n_layer=4, d_ff=192,
                 dropout=0.1, block_size=33, causal=False, pool='mean'):
        self.n_feat = n_feat
        self.d_model = d_model
        self.n_head = n_head
        self.n_layer = n_layer
        self.d_ff = d_ff
        self.dropout = dropout
        self.block_size = block_size       # max seq len (include [CLS] if used)
        self.causal = causal
        self.pool = pool                   # 'mean' or 'cls'


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        assert cfg.d_model % cfg.n_head == 0
        self.cfg = cfg
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.attn_drop = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        # this is the ONE line that makes it GPT vs BERT
        if cfg.causal:
            mask = torch.tril(torch.ones(cfg.block_size, cfg.block_size)).view(1, 1, cfg.block_size, cfg.block_size)
            self.register_buffer('mask', mask)

    def forward(self, x, return_attn: bool = False):
        B, L, D = x.shape
        H = self.cfg.n_head
        qkv = self.qkv(x).reshape(B, L, 3, H, D // H).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]                                  # (B, H, L, dh)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(D // H)               # (B, H, L, L)
        if self.cfg.causal:
            att = att.masked_fill(self.mask[:, :, :L, :L] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)
        y = (att @ v).transpose(1, 2).contiguous().view(B, L, D)
        y = self.resid_drop(self.proj(y))
        return (y, att) if return_attn else y


class Block(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.mlp = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.d_ff),
            nn.GELU(),
            nn.Linear(cfg.d_ff, cfg.d_model),
            nn.Dropout(cfg.dropout),
        )

    def forward(self, x, return_attn: bool = False):
        if return_attn:
            a, att = self.attn(self.ln1(x), return_attn=True)
            x = x + a
        else:
            x = x + self.attn(self.ln1(x)); att = None
        x = x + self.mlp(self.ln2(x))
        return (x, att) if return_attn else x


class Transformer(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_proj = nn.Linear(cfg.n_feat, cfg.d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, cfg.d_model)) if cfg.pool == 'cls' else None
        self.pos_emb = nn.Parameter(torch.zeros(1, cfg.block_size, cfg.d_model))
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, 1)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x, return_attn: bool = False):
        h = self.tok_proj(x)                               # (B, L, D)
        if self.cls_token is not None:
            h = torch.cat([self.cls_token.expand(h.size(0), -1, -1), h], dim=1)
        h = self.drop(h + self.pos_emb[:, : h.size(1)])
        atts = []
        for blk in self.blocks:
            if return_attn:
                h, a = blk(h, return_attn=True); atts.append(a)
            else:
                h = blk(h)
        h = self.ln_f(h)
        pooled = h[:, 0] if self.cfg.pool == 'cls' else h.mean(dim=1)
        logit = self.head(pooled).squeeze(-1)
        return (logit, atts) if return_attn else logit
# ============================================================================

print('nanoGPT-style model defined — line count:',
      sum(1 for _ in open(__file__)) if '__file__' in dir() else 'inline')

### 讨论：`causal: bool` 这一行的意义

| flag | 语义 | 典型用途 |
|------|------|---------|
| `True` | 每个位置只能看**过去** | GPT 自回归、序列预测、next-token / "预测下一笔" |
| `False` | 每个位置能看**整窗** | BERT / 异常检测 / 分类（整窗已观测） |

对"窗口最后一笔是否欺诈"——当交易已经发生、整窗历史可见，双向（`causal=False`）更合理。我们主实验用 `False`，末尾再做 `True` 对照。

## 3. 把 Kaggle CSV 变成时序窗口

Kaggle 原数据没有 `user_id`。构造一个 pseudo user：

```
user_id = hash(Amount 分桶 + Time / 3600 分桶) % 256
```

**然后按时间切分 70/15/15**：
- 边界 1: `T70 = quantile(Time, 0.70)`
- 边界 2: `T85 = quantile(Time, 0.85)`
- 训练用 `Time ≤ T70`，val `T70 < Time ≤ T85`，test `Time > T85`——确保未来信息绝不进训练。

In [ ]:
# ---- features ----
feat_cols = [c for c in df.columns if c not in ('Time', 'Class')]  # V1..V28 + Amount
df_sorted = df.sort_values('Time').reset_index(drop=True)

# pseudo user id
amount_bucket = pd.qcut(df_sorted['Amount'], q=16, labels=False, duplicates='drop')
hour_bucket = (df_sorted['Time'] // 3600).astype(int)
df_sorted['user_id'] = (amount_bucket * 131 + hour_bucket * 7) % 256

# standardize features on TRAIN ONLY (computed below after we know T70)
T70 = df_sorted['Time'].quantile(0.70)
T85 = df_sorted['Time'].quantile(0.85)
print(f'time boundaries: T70={T70:.0f}s  T85={T85:.0f}s  (span={df_sorted["Time"].max():.0f}s)')

train_mask = df_sorted['Time'] <= T70
scaler = StandardScaler().fit(df_sorted.loc[train_mask, feat_cols].values)
feats_all = scaler.transform(df_sorted[feat_cols].values).astype(np.float32)
print('feat_all shape', feats_all.shape)

In [ ]:
# ---- sliding windows per pseudo-user ----
L = 32  # window length (current + 31 history)

def build_windows(df_sorted, feats_all, L=32):
    X, y, t_end = [], [], []
    for uid, grp in df_sorted.groupby('user_id'):
        idx = grp.index.values
        if len(idx) < L:
            continue
        for end in range(L - 1, len(idx)):
            window = idx[end - L + 1: end + 1]
            X.append(feats_all[window])
            y.append(df_sorted.at[idx[end], 'Class'])
            t_end.append(df_sorted.at[idx[end], 'Time'])
    return (np.stack(X).astype(np.float32),
            np.array(y, dtype=np.float32),
            np.array(t_end, dtype=np.float64))

X, y, t_end = build_windows(df_sorted, feats_all, L=L)
print(f'windows: {X.shape}   positives: {int(y.sum())}   pos rate: {y.mean()*100:.4f}%')

In [ ]:
# ---- TIME-BASED split using the window's LAST-row time ----
tr_m = t_end <= T70
va_m = (t_end > T70) & (t_end <= T85)
te_m = t_end > T85

Xtr, ytr = torch.from_numpy(X[tr_m]), torch.from_numpy(y[tr_m])
Xva, yva = torch.from_numpy(X[va_m]), torch.from_numpy(y[va_m])
Xte, yte = torch.from_numpy(X[te_m]), torch.from_numpy(y[te_m])

for name, yy in [('train', ytr), ('val', yva), ('test', yte)]:
    print(f'{name}: n={len(yy):,}  pos={int(yy.sum())}  rate={yy.float().mean().item()*100:.4f}%')

**No-leakage check**：
- split 用的是 **窗口末尾时间**，所以训练样本的整个 32-row 窗口也在 `T70` 之前
- 但注意边界样本（`t_end ≤ T70` 但 `t_start` 在 T70 附近）是干净的；真正要防的是 val 窗口里**含**训练时段——这里不存在，因为 val 的 `t_end > T70` 只意味着最后一笔在之后，过去 31 笔本身是合法历史，不构成泄露。

## 4. 分类头 + Pooling 变体

已经在 `Transformer.cfg.pool` 里内置了 `mean` / `cls`。下面分别训练两个配置。

In [ ]:
# data loaders
def make_loader(X, y, bs=128, shuffle=True):
    return DataLoader(TensorDataset(X, y), batch_size=bs, shuffle=shuffle, drop_last=False)

tr_loader = make_loader(Xtr, ytr, bs=128, shuffle=True)
va_loader = make_loader(Xva, yva, bs=512, shuffle=False)
te_loader = make_loader(Xte, yte, bs=512, shuffle=False)
print('batches —', 'train', len(tr_loader), 'val', len(va_loader), 'test', len(te_loader))

## 5. Warmup 训练 + 早停

In [ ]:
def noam_lr(step, d_model, warmup):
    step = max(step, 1)
    return d_model ** -0.5 * min(step ** -0.5, step * warmup ** -1.5)

def eval_probs(model, loader):
    model.eval()
    ps, ys = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            p = torch.sigmoid(model(xb)).cpu().numpy()
            ps.append(p); ys.append(yb.numpy())
    return np.concatenate(ps), np.concatenate(ys)

def train(model, tr_loader, va_loader, *,
          epochs=15, warmup=1000, patience=3, tag='run'):
    d_model = model.cfg.d_model if hasattr(model, 'cfg') else 64
    pos_w = torch.tensor([max((ytr == 0).sum().item() / max((ytr == 1).sum().item(), 1), 1.0)],
                         device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda=lambda s: noam_lr(s + 1, d_model, warmup))
    best_ap, best_state, bad = -1.0, None, 0
    hist = {'train_loss': [], 'val_ap': []}
    for ep in range(1, epochs + 1):
        model.train(); tot, n = 0.0, 0
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
            tot += loss.item() * xb.size(0); n += xb.size(0)
        p, yy = eval_probs(model, va_loader)
        ap = average_precision_score(yy, p)
        hist['train_loss'].append(tot / n); hist['val_ap'].append(ap)
        print(f'[{tag}] ep{ep:02d}  loss={tot/n:.4f}  val_AUC-PR={ap:.4f}')
        if ap > best_ap + 1e-4:
            best_ap, best_state, bad = ap, {k: v.detach().clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience:
                print(f'[{tag}] early stop'); break
    model.load_state_dict(best_state)
    return model, hist, best_ap

In [ ]:
# ---- main run: bidirectional (causal=False), mean pooling ----
torch.manual_seed(SEED)
cfg_main = GPTConfig(n_feat=X.shape[-1], d_model=96, n_head=4, n_layer=4, d_ff=192,
                     dropout=0.1, block_size=L + 1, causal=False, pool='mean')
model_main = Transformer(cfg_main).to(device)
print('params:', sum(p.numel() for p in model_main.parameters()))
model_main, hist_main, ap_main = train(model_main, tr_loader, va_loader,
                                       epochs=12, warmup=1000, patience=3, tag='main')
print(f'>> main val AUC-PR = {ap_main:.4f}')

## 6. MLP baseline（同输入、同 split、公平对照）

把 (L=32, F) 序列 flatten → MLP；这样 Transformer 的优势完全来自"结构"，而不是数据。

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_in, hidden=(256, 128), dropout=0.2):
        super().__init__()
        layers = []
        last = n_in
        for h in hidden:
            layers += [nn.Linear(last, h), nn.ReLU(), nn.Dropout(dropout)]
            last = h
        layers.append(nn.Linear(last, 1))
        self.net = nn.Sequential(*layers)
        self.cfg = type('C', (), {'d_model': 64})()  # dummy for train() d_model

    def forward(self, x):
        B = x.size(0)
        return self.net(x.reshape(B, -1)).squeeze(-1)

torch.manual_seed(SEED)
mlp = MLP(n_in=L * X.shape[-1]).to(device)
print('MLP params:', sum(p.numel() for p in mlp.parameters()))
mlp, hist_mlp, ap_mlp = train(mlp, tr_loader, va_loader, epochs=12, warmup=500, patience=3, tag='mlp')
print(f'>> MLP val AUC-PR = {ap_mlp:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(hist_main['val_ap'], label=f'Transformer (best={ap_main:.3f})')
ax.plot(hist_mlp['val_ap'], label=f'MLP baseline (best={ap_mlp:.3f})')
ax.set_xlabel('epoch'); ax.set_ylabel('val AUC-PR'); ax.legend()
ax.set_title('Transformer vs MLP baseline — Kaggle fraud')
print(f'advantage: +{(ap_main - ap_mlp)*100:.2f} pp AUC-PR')

## 7. 测试集完整评估

In [ ]:
def full_eval(model, te_loader, name='model'):
    probs, ys = eval_probs(model, te_loader)
    ap = average_precision_score(ys, probs)
    auc = roc_auc_score(ys, probs)
    fpr, tpr, _ = roc_curve(ys, probs)
    m = fpr <= 0.001
    r_at_fpr = tpr[m].max() if m.any() else 0.0
    print(f'[{name}] TEST — AUC-PR={ap:.4f}  AUC-ROC={auc:.4f}  Recall@FPR=0.001={r_at_fpr:.4f}')
    return ap, probs, ys

ap_main_te, probs_te, ys_te = full_eval(model_main, te_loader, 'transformer')
ap_mlp_te, _, _ = full_eval(mlp, te_loader, 'mlp')
print(f'>> test Transformer - MLP = +{(ap_main_te - ap_mlp_te)*100:.2f} pp AUC-PR')

# confusion matrix at val-tuned best-F1 threshold
_, p_va, y_va = full_eval(model_main, va_loader, 'val')
prec, rec, th = precision_recall_curve(y_va, p_va)
f1 = 2 * prec * rec / (prec + rec + 1e-9)
best_t = th[np.nanargmax(f1[:-1])] if len(th) else 0.5
pred_te = (probs_te >= best_t).astype(int)
cm = confusion_matrix(ys_te, pred_te)
fig, ax = plt.subplots(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['pred 0','pred 1'], yticklabels=['true 0','true 1'])
ax.set_title(f'test CM @ t={best_t:.3f}')

## 8. 注意力可视化（一条欺诈窗口）

In [ ]:
pos_idx = (yte == 1).nonzero(as_tuple=True)[0]
if len(pos_idx) == 0:
    print('no fraud in test set (odd — check split)');
else:
    i = pos_idx[0].item()
    window = Xte[i].unsqueeze(0).to(device)
    model_main.eval()
    with torch.no_grad():
        logit, atts = model_main(window, return_attn=True)
    # atts: list of (B, H, L, L); take last layer, head 0
    A = atts[-1].squeeze(0).cpu().numpy()  # (H, L, L)
    fig, axes = plt.subplots(1, cfg_main.n_head, figsize=(3 * cfg_main.n_head, 3))
    for h, ax in enumerate(axes):
        ax.imshow(A[h], cmap='viridis')
        ax.set_title(f'L4 head{h}')
        ax.set_xlabel('key (history)'); ax.set_ylabel('query')
    fig.suptitle(f'Fraud window idx={i}  logit={logit.item():.2f}', y=1.02)
    print('look at the last row of each head — where does the model focus when predicting this tx?')

## 9. 对照实验：`causal=True` (GPT 风格)

同样的数据、同样的超参，只把 `cfg.causal` 翻一下。预期：AUC-PR 明显下降——因为异常检测看的是"窗口最后一笔是否异常"，双向能利用全部历史的聚合信息，而因果 mask 强制每个位置**不能看未来的历史**，相当于把能看到的上下文砍了一半。

In [ ]:
torch.manual_seed(SEED)
cfg_causal = GPTConfig(n_feat=X.shape[-1], d_model=96, n_head=4, n_layer=4, d_ff=192,
                       dropout=0.1, block_size=L + 1, causal=True, pool='mean')
model_causal = Transformer(cfg_causal).to(device)
model_causal, hist_causal, ap_causal = train(model_causal, tr_loader, va_loader,
                                             epochs=10, warmup=1000, patience=3, tag='causal')
print(f'>> causal=True val AUC-PR = {ap_causal:.4f}')
print(f'>> bidir val AUC-PR       = {ap_main:.4f}')
print(f'>> gap (bidir - causal)   = {(ap_main - ap_causal)*100:+.2f} pp')

### 结论表 — GPT (causal) vs BERT (bidirectional)

| 维度 | GPT (causal=True) | BERT (causal=False) |
|------|-------------------|---------------------|
| Mask | 下三角 `tril` | 无 mask（全 1） |
| 每位置看什么 | 仅自己 + 过去 | 整窗 |
| 适合任务 | 自回归生成、next-step 预测 | 分类、embedding、异常检测 |
| 可并行度 | 训练可并行、推理自回归 | 训推都可并行 |
| 本实验结果 | val AP ≈ `ap_causal` | val AP ≈ `ap_main` |
| 代码差异行数 | 3 行（`register_buffer` + `masked_fill`） | — |

**本实验选 `causal=False` 的理由**：
- 我们的 label 是"整窗是否在最后一行出现异常"，**整窗可见**在业务上成立（交易已发生）
- 双向能让 `query=L-1` 看到完整 32-笔历史；causal 里每个 `query_i` 只能看 `0..i`，相当于 32 次不同视角的聚合变弱

> 小练习：把 label 改成"预测窗口**下一步**是否异常"，causal=True 就回到正确方向了——这就是下周 W9 无监督预测式异常检测的起点。

## 10. 保存 checkpoint

In [ ]:
ckpt_path = PROJECT_ROOT / 'models' / 'transformer_v2.pt'
ckpt_path.parent.mkdir(exist_ok=True, parents=True)
torch.save({
    'state_dict': model_main.state_dict(),
    'cfg': vars(cfg_main),
    'val_ap': ap_main,
    'test_ap': ap_main_te,
    'feat_cols': feat_cols,
    'scaler_mean': scaler.mean_.tolist(),
    'scaler_scale': scaler.scale_.tolist(),
    'L': L,
    'time_boundaries': {'T70': float(T70), 'T85': float(T85)},
    'seed': SEED,
}, ckpt_path)
print('saved', ckpt_path, f'({ckpt_path.stat().st_size / 1024:.1f} KB)')

## 11. 本周复盘

**1. GPT vs BERT 在代码上的差异（精确到行）**：
- `CausalSelfAttention.__init__`: 多了 `register_buffer('mask', torch.tril(...))`
- `CausalSelfAttention.forward`: 多了 `att = att.masked_fill(self.mask[...] == 0, -inf)`
- 就这两行。其它结构（qkv proj / 多头拆分 / 残差 / LN / MLP）完全一致。

**2. Transformer 比 MLP 强在哪？**
- MLP 把序列 flatten → 丢掉了"顺序 / 位置"信息；每个特征的位置对它一视同仁
- Transformer 通过 PE + self-attention 保留了"第 i 笔 vs 第 j 笔"的相对关系，能学到"与过去某笔相似"这种模式

**3. 时间切分 vs 随机切分的坑**
- 随机切分会让同一天的交易分散到 train/val/test——val 上"似曾相识"的样本很多，AP 虚高
- 时间切分更真实地模拟"上线后遇到全新交易"的分布

**4. 下周 W9 预告**
- 把 Encoder 改成"重构"：输入序列 → 输出序列，MSE 损失
- 重构误差作为异常分数（无监督）
- 需要改的行：去掉分类头 → 加一个 `Linear(d_model, n_feat)` 输出层

---

**本周验收自检**：
- [ ] Transformer test AUC-PR **>** MLP test AUC-PR
- [ ] 时间切分无泄露（能说清 T70 / T85 两条边界）
- [ ] `causal=True` 对照实验已跑，能解释 3 行代码差异
- [ ] attention heatmap 已出图
- [ ] `models/transformer_v2.pt` 已保存